In [1]:
import torch
import torch.nn as nn
import pandas as pd
from transformers import BertTokenizer, BertModel

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [3]:
class CrossEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.bert = BertModel.from_pretrained("bert-base-uncased")
        self.linear = nn.Linear(self.bert.config.hidden_size, 1)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        cls_output = outputs.last_hidden_state[:, 0]
        score = self.linear(cls_output)
        return score.squeeze(-1)

In [ ]:
MODEL_PATH = r"C:\Users\nihal\Desktop\NLP_Project\cross_encoder_cpu.pt"

model = CrossEncoder().to(device)
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model.eval()


In [5]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
MAX_LEN = 128


In [6]:
BM25_PATH = r"C:\Users\nihal\Desktop\NLP_Project\bm25_subset_candidates.csv"

bm25_df = pd.read_csv(BM25_PATH)
bm25_df.head()


,query_id,query,passage_id,passage,bm25_score,rank
0,737889,what is decentralization process.,189333,ANSWERS TO FREQUENTLY ASKED QUESTIONS ON THE D...,20.736305,1
1,737889,what is decentralization process.,383602,What is EDI? Electronic Data Interchange (EDI)...,12.944016,2
2,737889,what is decentralization process.,547695,The concepts of centralization and decentraliz...,12.655926,3
3,737889,what is decentralization process.,142911,Response 2: Negotiation is a process civilized...,12.514990,4
4,737889,what is decentralization process.,454015,The Fluorescence Process. Fluorescence is the ...,11.644176,5


In [7]:
example_query = "what does an endocrinologist do"

candidates = bm25_df[bm25_df["query"] == example_query].sort_values("rank")
candidates.head(5)


,query_id,query,passage_id,passage,bm25_score,rank
11200,631853,what does an endocrinologist do,582928,What is an endocrinologist? An endocrinologist...,14.397651,1
11201,631853,what does an endocrinologist do,332128,Your regular gynecologist may do some basic te...,14.294946,2
11202,631853,what does an endocrinologist do,126780,What does an Advertising Manager do? An advert...,12.561770,3
11203,631853,what does an endocrinologist do,292336,An endocrinologist is a doctor who has studied...,12.544027,4
11204,631853,what does an endocrinologist do,493272,What Does A Logo Do. Just what does a logo do ...,12.319349,5


In [8]:
def encode_pair(query, passage):
    return tokenizer(
        query,
        passage,
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN,
        return_tensors="pt"
    )


In [9]:
top_candidates = candidates.head(10).copy()

scores = []

with torch.no_grad():
    for _, row in top_candidates.iterrows():
        enc = encode_pair(row["query"], row["passage"])

        input_ids = enc["input_ids"].to(device)
        attention_mask = enc["attention_mask"].to(device)

        score = model(input_ids, attention_mask)
        scores.append(score.item())

top_candidates["transformer_score"] = scores


In [10]:
print("BM25 ranking:")
display(top_candidates[["rank", "passage"]])


BM25 ranking:


,rank,passage
11200,1,What is an endocrinologist? An endocrinologist...
11201,2,Your regular gynecologist may do some basic te...
11202,3,What does an Advertising Manager do? An advert...
11203,4,An endocrinologist is a doctor who has studied...
11204,5,What Does A Logo Do. Just what does a logo do ...
11205,6,college Video What DMT Does To Your Body What ...
11206,7,What does suv stand for? Car keys. Suv meaning...
11207,8,"What Does Name Jashelle Mean You are honest, b..."
11208,9,WHAT DOES IEP STAND FOR? WHAT DOES DIS MEAN? I...
11209,10,Report Abuse. APO stands for Army Post Office....


In [11]:
import pandas as pd

pd.set_option("display.max_colwidth", None)   
pd.set_option("display.max_rows", 20)         
pd.set_option("display.max_columns", None)

In [12]:
print("Transformer re-ranking:")
display(
    top_candidates
    .sort_values("transformer_score", ascending=False)
    [["transformer_score", "passage"]]
)


Transformer re-ranking:


,transformer_score,passage
11203,3.021697,"An endocrinologist is a doctor who has studied the endocrine system and its diseases. These doctors know how to diagnose the diseases of the endocrine glands, and also how to treat them."
11200,2.784038,"What is an endocrinologist? An endocrinologist is a specially trained doctor who can diagnose and treat diseases that affect your glands, hormones and your endocrine system. The pancreas is part of the endocrine system, and insulin is one of the central hormones the body needs to function properly."
11201,2.416219,"Your regular gynecologist may do some basic testing. Or, you may be referred to a reproductive endocrinologist (a doctor specializing in fertility) or a urologist (for male infertility) for more thorough fertility testing. Fertility testing involves both partners."
11207,-1.551465,"What Does Name Jashelle Mean You are honest, benevolent, brilliant and often inventive, full of high inspirations. You are courageous, honest, determined, original and creative. You are a leader, especially for a cause. Sometimes you do not care to finish what you start, and may leave details to others."
11202,-3.541766,"What does an Advertising Manager do? An advertising manager will typically do the following: Work with department heads or staff to discuss topics such as contracts, selection of advertising media, or products to be advertised. Gather and organize information to plan advertising campaigns."
11205,-3.575433,"college Video What DMT Does To Your Body What DMT Does To Your Body 1:21 What DMT Does To Your Body DMT is a psychedelic drug that can effect your body in strange and scary ways. Check out this video to learn what DMT does to your body. Transcript: DMT creates an almost immediate sense of euphoria, a feeling of sheer relaxation, and peaceful hallucinations.... DMT creates an almost immediate sense of euphoria, a feeling of sheer relaxation, and peaceful hallucinations."
11204,-3.777144,"What Does A Logo Do. Just what does a logo do for your website? It gives visitors the first impression of your website. Ideally, your logo will represent the essence of your site, whether it's classy, serious, business-like, or off-beat. A website logo also helps contribute a consistent look and feel throughout all the pages of a website. An important consideration when building your website is the design of your logo."
11208,-4.002915,"WHAT DOES IEP STAND FOR? WHAT DOES DIS MEAN? IEP is an acronym for Individualized Education Program, which is a unique document required by the government that aids the ability of a disabled student to receive a quality education."
11209,-4.367450,"Report Abuse. APO stands for Army Post Office. The AE stands for Army Europe and the number are just a place state side that the mail will be shipped from to get to the location overseas. It does it this way so it does not cost as much to ship things then it would with regular overseas mail.E: What does APO, AE 09356 mean? what does it mean? i know its a military address but what does AE stand for?"
11206,-4.438389,"What does suv stand for? Car keys. Suv meaning, definition, what is suv abbreviation for sport utility vehicle a large car with an engine that supplies power to. What does suv stand for in car? All acronyms dictionarycrossover what's the difference? Auto trader. Vwhat does suv stand for? I know it's a type of car. Suv definition by acronymfinder."
